## 1. Hypothesis Generation and Verstehen

__Conversations at Scale: Robust AI-led Interviews (https://ssrn.com/abstract=4974382)__

Interviews relate to a tradition in the social sciences emphasising the importance of understanding people from their own subjective point of view (e.g., the “Verstehen” approach in Weber [1925], Dilthey [1884]). For AI-led interviews in particular, studies may result in hundreds of transcripts which can be challenging to analyse manually. This notebook `01-hypothesis-generation.ipynb` allows you to have a conversation with a language model about a large sample of interview transcripts from a study. This can help to obtain an overview of interview contents and to generate hypotheses relatively quickly. Based on the generated hypotheses, the second notebook `02-labelling.ipynb` allows to analyse each transcript separately and to measure more precisely how frequently certain themes or topics occur. For further discussion, please refer to Section 2.3 of the paper. <br>
For example, in the meaning study of the paper, we learned via language model-aided hypothesis generation as illustrated in this notebook that pets were one topic commonly discussed in the transcripts. We then used code such as `02-labelling.ipynb` to measure in detail which fraction of all transcripts contained mentions of pet care as a source of meaning in life.

Before running this notebook, please follow the README.md file on how to set up VS Code for running Jupyter notebooks, as well as creating the `interviews` conda environment with all necessary libraries. Connect to this environment in VS Code by clicking on "Select Kernel" in the top right corner and selecting `interviews`. Afterwards, open the file `details.py` and paste your API keys. __Ensure that the file `details.py` containing your API keys is not accidentally shared in a public repository.__ So far the code supports the OpenAI, Anthropic, Google, and Microsoft Azure APIs. The underlying functions in `core.py` can be extended relatively easily to support other language model APIs as well, or, if sufficient local compute is available, the code could be reworked to run with local language models.

__Note:__ In the following examples, reasoning efforts are disabled or set to low to allow for faster responses and lower costs, but you may find more accurate responses if you set them to high (assuming a given language model supports reasoning).

To start, run the cells below step-by-step.

In [ ]:
import core
import details

import random

First, set the `transcript_directory` variable to the directory where the interview transcripts are stored. The code below works with the transcript text files which the interview platform saves at the end of interviews.

The next cell combines all transcripts and computes the *approximate* number of total tokens (a token is ca. 3/4 of a word with common current tokenisers). If this number exceeds the context window of the language model you plan to use, consider sampling a smaller fraction of the transcripts (by setting `fraction<1`), using a model with a larger context window, or summarising each transcript with a separate language model first.

In [ ]:
# Load transcripts
transcript_directory = "..."
transcript_paths = core.load_file_paths(transcript_directory)
transcripts = []
for path in transcript_paths:
    with open(path, "r", encoding="utf-8") as f:
        transcripts.append(f.read())

# Adjust fraction of transcripts to use
random.seed(42)
fraction = 1.0
transcripts = random.sample(transcripts, int(len(transcripts) * fraction))

# Concatenate transcripts with separator
summaries_str = "\n\n---\n\n".join(transcripts)
combined_interview_transcripts = f"""The following are {len(transcripts)} interview transcripts, separated by '---':


{summaries_str}"""

print(
    f"Note: The current prompt contains {int(len(transcripts))} transcripts and has around {len(combined_interview_transcripts.split())} words, or approximately {int((4/3)*len(combined_interview_transcripts.split()))} tokens. If this exceeds model's context length, sample a smaller fraction of transcripts."
)

Next, adjust the following system prompt which will guide the language model in analysing the interview transcripts. It can be helpful to ask the model to impersonate an expert in the research field of the interview transcripts, in order to ask about surprising findings subsequently. For example, for the educational/occupational choice study in the paper, the system prompt could be `system = "You are an professor of economics at a leading university. Your field is labour economics and you focus on studying educational and occupational choices."`

In [ ]:
system = "..."

The next cells allow chatting with different language models about the content of the interview transcripts from your study. For additional details on the input parameters of the `verstehen` function, see the docstring and code in the module `core.py`. __To begin a chat, run one of the following cells, type your first question, and confirm with Enter.__

From our experience, good first questions to ask the LLM are for example:

- Please briefly summarise the most salient themes, citing evidence from the transcripts.

- From your knowledge of the literature, which findings in the transcripts do you find particularly interesting or surprising? Please cite specific examples.

<br>

__OpenAI API__

The reasoning effort can also be set to values such as `low`, `medium`, or `high`. For other API arguments which can be set in `additional_api_kwargs`, see the documentation of the API https://platform.openai.com/docs/api-reference/introduction.

In [ ]:
from openai import OpenAI

core.verstehen(
    combined_interview_transcripts,
    system=system,
    model="gpt-5.2-2025-12-11",
    client=OpenAI(api_key=details.KEY_OPENAI),
    additional_api_kwargs={"reasoning": {"effort": "none"}},
)

__Anthropic API__

Thinking mode can be `enabled`, but requires setting e.g. {"budget_tokens": 10000} to specify the budget for thinking tokens generated by the model; for other API arguments which can be set in `additional_api_kwargs`, see the documentation of the API https://docs.claude.com/en/api/overview.

In [ ]:
import anthropic

core.verstehen(
    combined_interview_transcripts,
    system=system,
    model="claude-sonnet-4-5-20250929",
    client=anthropic.Anthropic(api_key=details.KEY_ANTHROPIC),
    additional_api_kwargs={
        "max_tokens": 2000,
        "temperature": 0.0,
        "thinking": {"type": "disabled"},
    },
)

__Google API__

The thinking level can be set to values such as `minimal`, `low`, `medium`, or `high` depending on the model used. For other API arguments which can be set in `additional_api_kwargs`, see the documentation of the API https://ai.google.dev/gemini-api/docs.

In [ ]:
from google import genai

core.verstehen(
    combined_interview_transcripts,
    system=system,
    model="gemini-3-flash-preview",
    client=genai.Client(api_key=details.KEY_GOOGLE),
    additional_api_kwargs={
        "config": {
            "max_output_tokens": 2000,
            "thinking_config": {"thinking_level": "minimal"},
        }
    },
)

__Microsoft Azure Inference API__

This API is one example of many APIs which offer access to a range of open-source models. A new resource can be created at https://ai.azure.com and language models such as e.g. Llama or DeepSeek be selected. Beyond the key, the endpoint and version have to be added to `details.py` as well. For API arguments which can be set in `additional_api_kwargs`, see the API documentation https://learn.microsoft.com/en-us/rest/api/aifoundry/modelinference/.

In [ ]:
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential

core.verstehen(
    combined_interview_transcripts,
    system=system,
    model="DeepSeek-V3-0324",
    client=ChatCompletionsClient(
        endpoint=details.ENDPOINT_AZURE,
        credential=AzureKeyCredential(details.KEY_AZURE),
        api_version=details.VERSION_AZURE,
    ),
    additional_api_kwargs={"max_tokens": 2000, "temperature": 0.0},
)